# Hands-On Machine Learning — Chapter 9
## Unsupervised Learning Techniques


### Introduction

In supervised learning, models are trained on labeled data to learn the mapping between input features and output labels. In **unsupervised learning**, there are no labels; the system attempts to discover structure, patterns, or groupings within the data itself.

Typical tasks include:
- **Clustering:** grouping similar instances together (e.g., customer segmentation).
- **Dimensionality Reduction:** simplifying data while preserving key information.
- **Anomaly Detection:** identifying rare or unusual data points.
- **Density Estimation:** modeling the probability distribution of data.

<p align="left"><img src="../fig/figure9.1.png" width="45%"></p>

Unsupervised learning reveals hidden patterns and often serves as a **preprocessing step** for supervised learning (e.g., clustering users before personalized recommendation).

### Clustering

Clustering algorithms aim to group data points based on similarity. Unlike classification, there are no predefined categories. Algorithms differ in how they define similarity and clusters:

- **Partitioning methods:** divide the dataset into non-overlapping clusters (e.g., K-Means).
- **Hierarchical methods:** build nested clusters (e.g., Agglomerative Clustering).
- **Density-based methods:** find clusters of high density separated by low-density regions (e.g., DBSCAN).
- **Model-based methods:** assume data is generated by a mixture of probabilistic models (e.g., Gaussian Mixture Models).

<p align="left"><img src="../fig/figure9.2.png" width="45%"></p>

### K-Means

The **K-Means algorithm** is one of the most popular clustering techniques. It aims to partition *n* observations into *k* clusters where each observation belongs to the cluster with the nearest mean (centroid).

#### Algorithm Steps:
1. Choose the number of clusters *k*.
2. Initialize *k* centroids randomly.
3. Assign each data point to its nearest centroid.
4. Compute new centroids as the mean of assigned points.
5. Repeat steps 3–4 until centroids stabilize or a maximum number of iterations is reached.

The objective function (to minimize) is the **inertia** (sum of squared distances from each point to its cluster centroid):

$$ J = \sum_{i=1}^{m} \sum_{j=1}^{k} r_{ij} ||x_i - \mu_j||^2 $$

where \(r_{ij}\) is 1 if instance *i* belongs to cluster *j*, otherwise 0.

<p align="left"><img src="../fig/figure9.3.png" width="45%"></p>

K-Means tends to find spherical clusters of similar size. It is efficient but sensitive to outliers and initial centroid placement.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
import matplotlib.pyplot as plt

# Generate synthetic 2D data
X, y = make_blobs(n_samples=500, centers=4, cluster_std=0.6, random_state=42)

# Fit K-Means model
kmeans = KMeans(n_clusters=4, random_state=42)
kmeans.fit(X)

y_pred = kmeans.predict(X)

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=y_pred, cmap='viridis', s=30)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], c='red', marker='x', s=200)
plt.title('K-Means Clustering Result')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()

### K-Means Optimization and K-Means++ Initialization

K-Means is sensitive to the initial placement of centroids. Poor initialization can lead to suboptimal clustering or slow convergence. To mitigate this, the **K-Means++** algorithm improves initialization by spreading initial centroids apart.

**K-Means++ steps:**
1. Choose one centroid uniformly at random from the data points.
2. For each data point *x*, compute its squared distance *D(x)* from the nearest centroid already chosen.
3. Choose the next centroid randomly with probability proportional to *D(x)^2*.
4. Repeat until *k* centroids are chosen.

This helps avoid poor local minima and speeds up convergence.

<p align="left"><img src="../fig/figure9.4.png" width="45%"></p>

### Inertia and Convergence

The **inertia** (within-cluster sum of squares) measures how internally coherent clusters are. The algorithm iteratively reduces inertia until convergence.

However, minimizing inertia does not necessarily lead to meaningful clusters—it only guarantees compact ones.

Convergence is declared when centroids stop moving significantly or after a maximum number of iterations.

<p align="left"><img src="../fig/figure9.5.png" width="45%"></p>

In [ ]:
# Visualize convergence behavior for multiple initializations
inertias = []
k_values = range(1, 10)

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(X)
    inertias.append(km.inertia_)

plt.plot(k_values, inertias, marker='o')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.show()

### Limitations of K-Means

K-Means works best under specific assumptions:
- Clusters are approximately spherical and of similar size.
- Features are continuous and use Euclidean distance as a similarity metric.

Limitations include:
- Poor performance on non-spherical or overlapping clusters.
- Sensitive to outliers (a single outlier can significantly affect centroids).
- Requires specifying the number of clusters *k* in advance.

<p align="left"><img src="../fig/figure9.6.png" width="45%"></p>

### Mini-Batch K-Means

For very large datasets, the standard K-Means algorithm becomes computationally expensive because it must process all data at once. The **Mini-Batch K-Means** algorithm solves this by using small random batches of data to update centroids incrementally.

**Advantages:**
- Much faster than full K-Means for large datasets.
- Approximate results are typically close to those of standard K-Means.

**Algorithm outline:**
1. Sample a small batch of data.
2. Assign samples to nearest centroids.
3. Update centroids using only the sampled data.
4. Repeat until convergence.

<p align="left"><img src="../fig/figure9.7.png" width="45%"></p>

In [ ]:
from sklearn.cluster import MiniBatchKMeans

mbk = MiniBatchKMeans(n_clusters=4, random_state=42, batch_size=50)
mbk.fit(X)

y_mbk = mbk.predict(X)

plt.scatter(X[:, 0], X[:, 1], c=y_mbk, cmap='viridis', s=30)
plt.scatter(mbk.cluster_centers_[:, 0], mbk.cluster_centers_[:, 1], c='red', marker='x', s=200)
plt.title('Mini-Batch K-Means Clustering Result')
plt.show()

### Finding the Optimal Number of Clusters

Determining the appropriate number of clusters *k* is one of the key challenges in K-Means. Two common approaches are:

#### 1. The Elbow Method
Plot the inertia as a function of *k*. The curve typically decreases rapidly and then levels off. The *elbow point* (where inertia reduction slows down) suggests a suitable value for *k*.

<p align="left"><img src="../fig/figure9.8.png" width="45%"></p>

#### 2. Silhouette Score
Measures how well each data point fits within its assigned cluster versus other clusters:

$$ s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))} $$

where:
- \(a(i)\): average intra-cluster distance for instance *i*.
- \(b(i)\): smallest average distance from *i* to instances in another cluster.

The Silhouette Score ranges from -1 to 1. Higher values indicate better-defined clusters.

In [ ]:
from sklearn.metrics import silhouette_score

best_k = None
best_score = -1

for k in range(2, 10):
    model = KMeans(n_clusters=k, random_state=42)
    labels = model.fit_predict(X)
    score = silhouette_score(X, labels)
    if score > best_score:
        best_score = score
        best_k = k

print(f'Best number of clusters (silhouette): {best_k}, Score = {best_score:.3f}')

### DBSCAN

**DBSCAN (Density-Based Spatial Clustering of Applications with Noise)** defines clusters as high-density regions separated by low-density areas. Unlike K-Means, it does not require specifying the number of clusters in advance and can identify outliers.

#### Key Concepts
- **ε (epsilon):** neighborhood radius.
- **min_samples:** minimum number of points required to form a dense region.
- **Core points:** have at least `min_samples` neighbors within ε.
- **Border points:** within ε of a core point but not dense enough themselves.
- **Noise points:** not part of any cluster.

<p align="left"><img src="../fig/figure9.9.png" width="45%"></p>

Advantages:
- Automatically determines number of clusters.
- Handles arbitrarily shaped clusters.
- Robust to outliers.

Limitations:
- Struggles with varying density clusters.
- Requires tuning ε and min_samples carefully.

In [ ]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=0.3, min_samples=5)
db_labels = dbscan.fit_predict(X)

plt.scatter(X[:, 0], X[:, 1], c=db_labels, cmap='plasma', s=30)
plt.title('DBSCAN Clustering')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()

### Comparing K-Means and DBSCAN

| Property | K-Means | DBSCAN |
|-----------|---------|--------|
| Requires number of clusters | Yes | No |
| Handles irregular shapes | Poorly | Excellent |
| Handles outliers | Poorly | Excellent |
| Computational cost | O(n * k * d) | O(n log n) |

<p align="left"><img src="../fig/figure9.10.png" width="45%"></p>

In practice, DBSCAN works best for well-separated dense clusters, while K-Means is preferable for convex, spherical clusters.

### Gaussian Mixture Models (GMM)

**Gaussian Mixture Models (GMM)** are a probabilistic approach to clustering. Each cluster is modeled as a Gaussian distribution, and the overall model is a weighted sum of these distributions:

$$ p(x) = \sum_{k=1}^K \pi_k \mathcal{N}(x | \mu_k, \Sigma_k) $$

where:
- \(\pi_k\) = weight of cluster k (mixing coefficient)
- \(\mu_k\), \(\Sigma_k\) = mean and covariance of Gaussian k

Unlike K-Means, GMM provides **soft clustering**, assigning each point a probability of belonging to each cluster.

<p align="left"><img src="../fig/figure9.11.png" width="45%"></p>

In [ ]:
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=4, covariance_type='full', random_state=42)
gmm.fit(X)
probs = gmm.predict_proba(X)
y_gmm = gmm.predict(X)

plt.scatter(X[:, 0], X[:, 1], c=y_gmm, cmap='viridis', s=30)
plt.title('Gaussian Mixture Model Clustering')
plt.show()

### Expectation-Maximization (EM) Algorithm

GMMs are trained using the **Expectation-Maximization (EM)** algorithm, which iteratively estimates cluster assignments and parameters:

1. **E-step:** Estimate probability that each instance belongs to each cluster.
2. **M-step:** Update the parameters (means, covariances, and weights) to maximize likelihood.
3. Repeat until convergence.

<p align="left"><img src="../fig/figure9.12.png" width="45%"></p>

This approach generalizes K-Means: when covariances are identical and spherical, GMM behaves similarly to K-Means.

### Selecting the Number of Components in GMM

To select an appropriate number of Gaussian components, use model selection criteria like:

- **Akaike Information Criterion (AIC):**
  $$ AIC = 2k - 2\ln(L) $$
- **Bayesian Information Criterion (BIC):**
  $$ BIC = k \ln(n) - 2\ln(L) $$

where *L* is the likelihood of the model and *k* is the number of parameters.

Smaller AIC/BIC values indicate a better model balance between goodness of fit and complexity.

In [ ]:
lowest_bic = np.infty
bic_values = []
n_components_range = range(1, 10)

for n in n_components_range:
    gmm = GaussianMixture(n_components=n, random_state=42)
    gmm.fit(X)
    bic = gmm.bic(X)
    bic_values.append(bic)
    if bic < lowest_bic:
        lowest_bic = bic

plt.plot(n_components_range, bic_values, marker='o')
plt.xlabel('Number of components')
plt.ylabel('BIC')
plt.title('Selecting Optimal GMM Components (BIC)')
plt.show()

### Anomaly Detection with GMM

Because GMM provides probabilistic scores, it can also be used for **anomaly detection**. Points with very low probability under the model are considered outliers.

<p align="left"><img src="../fig/figure9.13.png" width="45%"></p>

Steps:
1. Fit GMM on training data.
2. Compute log-likelihood of each point.
3. Define a threshold below which points are anomalies.

In [ ]:
log_probs = gmm.score_samples(X)
threshold = np.percentile(log_probs, 2)  # bottom 2%
anomalies = X[log_probs < threshold]

plt.scatter(X[:, 0], X[:, 1], alpha=0.3)
plt.scatter(anomalies[:, 0], anomalies[:, 1], color='red', label='Anomalies')
plt.legend()
plt.title('Anomaly Detection using GMM')
plt.show()

### Hierarchical Clustering (Optional)

Hierarchical clustering builds a hierarchy of clusters using **agglomerative** (bottom-up) or **divisive** (top-down) approaches. It is visualized using a **dendrogram**.

<p align="left"><img src="../fig/figure9.14.png" width="45%"></p>

Agglomerative clustering merges the two closest clusters iteratively until only one cluster remains.

Linkage criteria determine how distances between clusters are computed:
- **Single linkage:** minimum distance between points.
- **Complete linkage:** maximum distance between points.
- **Average linkage:** average distance between all pairs.
- **Ward’s method:** minimizes within-cluster variance.

### Practical Guidelines

**When to use which clustering algorithm:**

| Situation | Recommended Method |
|------------|-------------------|
| Spherical clusters, speed important | K-Means or MiniBatchKMeans |
| Arbitrary shapes, presence of noise | DBSCAN |
| Overlapping distributions | GMM |
| Nonlinear manifold structure | Hierarchical or Spectral Clustering |

**Tips:**
- Always standardize features before clustering.
- Use PCA for dimensionality reduction before clustering high-dimensional data.
- Evaluate multiple runs and random seeds for stability.
- Combine clustering with visualization (e.g., PCA or t-SNE) for interpretation.

Clustering results depend heavily on data scaling, distance metrics, and initialization. In practice, unsupervised learning is often used for exploration rather than definitive conclusions.